In [0]:
source_dir="/Volumes/eccom_catalog/default/ecomm_data/inventory_data/Source/"
archive_dir="/Volumes/eccom_catalog/default/ecomm_data/inventory_data/Archive/"
inventory_stage_table="eccom_catalog.default.inventory_stage"
error_table="eccom_catalog.default.error_table"

In [0]:
#Read inventory csv file
from pyspark.sql.functions import col,lit,current_timestamp,datediff,current_date,when
from datetime import datetime
try:
	df_inventory=spark.read.csv(source_dir, header=True, inferSchema=True, dateFormat="yyyy-MM-dd", timestampFormat="yyyy-MM-dd HH:mm:ss")
	df_inventory=df_inventory.withColumn("processed_timestamp", current_timestamp()).withColumn("batch_id", lit(datetime.now().strftime("%Y%m%d_%H%M%S"))).withColumn("source_system", lit("ecommerce_inventory"))
	print("Successfully read inventory file")
except Exception as e:
	print(f"Failed to read inventory file: {str(e)}")
	raise

In [0]:
#Data Quality checks
try:
	total_records=df_inventory.count()
	total_null_inventory_ids=df_inventory.filter(col("inventory_id").isNull()).count()
	total_negative_stock=df_inventory.filter(col("stock_quantity")<0).count()	
	total_negative_reserved=df_inventory.filter(col("reserved_quantity")<0).count()
	total_invalid_available=df_inventory.filter(col("available_quantity")<0).count()

	print(f"Total records: {total_records}")
	print(f"Total null inventory ids: {total_null_inventory_ids}")
	print(f"Total negative stock: {total_negative_stock}")
	print(f"Total negative reserved: {total_negative_reserved}")
	print(f"Total invalid available quantity: {total_invalid_available}")
	print("Successfully applied Data quality checks")

	#Filter out valid records
	valid_inventory=df_inventory.filter((col("inventory_id").isNotNull()) & (col("stock_quantity")>=0) & (col("reserved_quantity")>=0) & (col("available_quantity")>=0))
	#Capture invalid records for error handling
	invalid_inventory=df_inventory.filter((col("inventory_id").isNull()) | (col("stock_quantity")<0) | (col("reserved_quantity")<0) | (col("available_quantity")<0))
	total_valid_inventory=valid_inventory.count()
	total_invalid_inventory=invalid_inventory.count()
	print(f"Total valid inventory: {total_valid_inventory}")
	print(f"Total invalid inventory: {total_invalid_inventory}")

except Exception as e:
	print(f"Failed to applied data quality checks: {str(e)}")
	raise

In [0]:
#Data enrichment - Inventory analytics
try:
	inventory=valid_inventory.withColumn("stock_utilization_rate", when(col("stock_quantity")>0, col("reserved_quantity")/col("stock_quantity"))
																   .otherwise(0))
	#Create stock status categories
	inventory=inventory.withColumn("stock_status", when(col("available_quantity")==0, "Out of Stock")
												   .when(col("available_quantity")<=col("reorder_level"), "Reorder Required")
												   .when(col("available_quantity")<=(col("reorder_level")*2), "Low Stock") 
												   .otherwise("In Stock"))	
	#Calculate days since restock
	inventory=inventory.withColumn("days_since_restock", datediff(current_date(),col("last_restocked")))
	#Calculate days since last audit
	inventory=inventory.withColumn("days_since_audit", datediff(current_date(), col("last_audit")))
	#Create audit status
	inventory=inventory.withColumn("audit_status", when(col("days_since_audit")>90, "Overdue")
												   .when(col("days_since_audit")>60, "Due Soon") 
												   .otherwise("Current"))
	print("Data enrichment completed")

except Exception as e:
	print(f"Failed to applied data enrichment: {str(e)}")
	raise 

In [0]:
try:
	#Write valid records to staging table
	inventory.write.format("delta").mode("overwrite").saveAsTable(inventory_stage_table)
	print("Successfully write inventory data to staging table")
	#Write invalid records to error table if there are any invalid records
	if total_invalid_inventory>0:
		invalid_inventory.write.format("delta").mode("append").saveAsTable(error_table)
		print("Total {total_invalid_inventory} invalid records write in error table")
	else:
	    print("No invalid records found")

except Exception as e:
	print(f"Failed to write data in inventory staging table: {str(e)}")
	raise

In [0]:
#Archive processed files
try:
	files=dbutils.fs.ls(source_dir)
	archive_count=0
	for file in files:
		if file.name.endswith('.csv'):
			source=file.path
			archive=archive_dir+file.name
			dbutils.fs.mv(source,archive)
			archive_count+=1
			print(f"Archived file: {file.name}")
	print(f"Successfully archived {archive_count} files")

except Exception as e:
	print(f"Error archiving files: {str(e)}")
	raise

In [0]:
#Log processing summary	
import json
try:
	processing_summary={
	"task": "inventory_stage_table",
	"timestamp": datetime.now().isoformat(),
	"total_records": total_records,
	"valid_records": total_valid_inventory,
	"invalid_records": total_invalid_inventory,
	"status": "SUCCESS" if total_invalid_inventory==0 else "SUCCESS_WITH_WARNINGS",
	"archived_files": archive_count
	}
	print(f"processing summary: {json.dumps(processing_summary)}")

	processing_log=spark.createDataFrame([processing_summary])
	processing_log.write.format("delta").mode("append").saveAsTable("eccom_catalog.default.processing_log")
	print("logged archived files")

except Exception as e:
	print(f"Failed to logged archived files")
	raise